## Example 1: quote identifiers and literals for SQL text

In [ ]:
# Example 1: identifier() double-quotes and escapes names; literal() renders Python values as SQL literals
from pg8000.native import identifier, literal

assert identifier("my table") == '"my table"'
assert identifier('odd"name') == '"odd""name"'
assert literal("it's") == "'it''s'"
assert literal(None) == "NULL" and literal(True) == "TRUE"
assert literal([1, 2, 3]) == "'{1,2,3}'"
print("Example 1:", identifier("my table"), literal("it's"), literal([1, 2, 3]))

## Example 2: named parameters become PostgreSQL's numbered placeholders

In [ ]:
# Example 2: to_statement rewrites :name parameters to $1, $2 ... and returns a function that orders the values
from pg8000.native import to_statement

sql, make_vals = to_statement("SELECT * FROM t WHERE a = :a AND b = :b OR a = :a")
assert sql == "SELECT * FROM t WHERE a = $1 AND b = $2 OR a = $1", sql
assert make_vals({"a": 10, "b": "x"}) == (10, "x")
print("Example 2:", sql, make_vals({"a": 10, "b": "x"}))

## Example 3: the text converters PostgreSQL's wire format needs

In [ ]:
# Example 3: parse PostgreSQL's text output for arrays, intervals and timestamps, and render Python values back
import datetime
from pg8000 import converters

assert converters.int_array_in("{1,2,NULL,4}") == [1, 2, None, 4]
assert converters.string_array_in('{"a b",c}') == ["a b", "c"]
assert converters.timestamp_in("2026-11-01 12:30:45.5") == datetime.datetime(2026, 11, 1, 12, 30, 45, 500000)
assert converters.date_out(datetime.date(2026, 11, 1)) == "2026-11-01"
print("Example 3:", converters.int_array_in("{1,2,NULL,4}"), converters.timestamp_in("2026-11-01 12:30:45.5"))

## Example 4: ranges and intervals as Python objects

In [ ]:
# Example 4: Range models PostgreSQL range types; PGInterval parses an interval string into its parts
from pg8000.native import PGInterval, Range

r = Range(1, 10, bounds="[)")
assert r.lower == 1 and r.upper == 10 and not r.is_empty
interval = PGInterval.from_str("1 year 2 mons 3 days 04:05:06")
assert (interval.years, interval.months, interval.days) == (1, 2, 3), interval
assert (interval.hours, interval.minutes, interval.seconds) == (4, 5, 6), interval
print("Example 4:", r, "|", interval)

## Example 5: a connection to a closed port is refused cleanly

In [ ]:
# Example 5: with no server listening, connect raises pg8000's InterfaceError instead of hanging
import socket
import pg8000.native

with socket.socket() as probe:
    probe.bind(("127.0.0.1", 0))
    port = probe.getsockname()[1]  # a port nothing listens on once the socket closes
try:
    pg8000.native.Connection("postgres", password="x", host="127.0.0.1", port=port, timeout=5)
except pg8000.native.InterfaceError as error:
    print("Example 5: refused as expected:", str(error)[:90])
else:
    raise AssertionError("a connection to a closed port succeeded")